In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
   전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}

table td.
</style>
"""))


**<font size="6" color="red">ch1. 허깅페이스 모델 사용</font>**
- Inference API 이용 : 모델의 결과를 server에서
- pipeline() 이용 : 모델을 다운로드 받아 모델의 결과를 local에서 

- 허깅페이스 transformer에서 지원하는 task
<div align="left">
| task값 | 설명 |
| :--- | :--- |
| text-classification (별칭 sentiment-analysis) |	감정 분석, 뉴스 분류, 리뷰 분류 등 문장 분류 |
| zero-shot-classification | 레이블에 대한 별도 학습 없이 후보 레이블 중에서 분류 |
| text-generation	| GPT 계열 모델을 이용한 텍스트 생성 |
| fill-mask |	문장 안의 빈칸(마스크)에 들어갈 단어 예측 |
| ner (token-classification의 별칭) |	개체명 인식(사람, 조직, 장소 등 라벨링) |
| question-answering |	주어진 지문(context)을 근거로 질문에 답변 |
| summarization	| 긴 문서를 짧게 요약 |
| translation |	서로 다른 언어 간 번역 |
| image-to-text | 이미지 내용을 설명하는 문장 생성 |
| image-classification | 이미지가 어떤 대상인지 분류 |
</div>

- 처음 모델 사용시 "c:/Users/내컴퓨터이름/.cache/huggingface"에 다운로드되느라 시간이 걸림

In [2]:
import warnings
import os
import logging
 
# 경고 메시지 제거
warnings.filterwarnings('ignore')
 
# transformers 라이브러리의 로깅 레벨을 ERROR로 조정 (경고 숨김)
logging.getLogger("transformers").setLevel(logging.ERROR)
 
# Hugging Face 캐시 관련 symlink 경고 제거
# os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# 1. 텍스트 기반 감정분석(긍정/부정)
- 토큰화 -> 워드임베딩 -> 모델 -> predict : pipeline()함수는 이 단계를 내부적으로 해줌

In [3]:
from transformers import pipeline
classifier = pipeline(task="text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
classifier("I've been waiting for a Hugging face course my whole life.")

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9982948899269104}]

In [4]:
# 특정 모델의 파라미터와 용량
from transformers import AutoModel
model = AutoModel.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")

# 전체 파라미터 수
total_params = sum(p.numel() for p in model.parameters() )
print(f"전체 파라미터 수: {total_params:,}")
print(f"전체 파라미터 수: {total_params/1024/1024:.3f}MB")

전체 파라미터 수: 66,362,880
전체 파라미터 수: 63.289MB


In [5]:
from transformers import pipeline
classifier = pipeline(task="sentiment-analysis",
                     model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

# 감정분석할 내용이 많으면 list'
classifier([
    "I've been waiting for a Hugging face course my whole life.",
    "I hate this so much!"
])

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9982948899269104},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [6]:
classifier("이 영화 정말 최고였어요. 감동적이고 연기도 대단해요!")

[{'label': 'POSITIVE', 'score': 0.975276529788971}]

In [7]:
classifier(["I like you", "I hate you", "힘들어요"])

[{'label': 'POSITIVE', 'score': 0.9998695850372314},
 {'label': 'NEGATIVE', 'score': 0.9991129040718079},
 {'label': 'POSITIVE', 'score': 0.8669533729553223}]

In [8]:
classifier = pipeline(task="sentiment-analysis",
                     model="daekeun-ml/koelectra-small-v3-nsmc")
texts = ['어찌 범부가 황새의 생각을 알겠습니까', '내가 울어도 파도는 밀려온다', '권선징악', '사랑시 고백구 행복동', '너가 좋아']
classifier(texts)

Device set to use cpu


[{'label': '1', 'score': 0.9794293642044067},
 {'label': '1', 'score': 0.8904697299003601},
 {'label': '1', 'score': 0.9947885274887085},
 {'label': '1', 'score': 0.9992324113845825},
 {'label': '1', 'score': 0.5141964554786682}]

In [9]:
for text, result in zip(texts, classifier(texts)):
    label = "긍정" if result['label']=='1' else "부정"
    print(f"'{text}' -> {label} {result['score']:.2%}")

'어찌 범부가 황새의 생각을 알겠습니까' -> 긍정 97.94%
'내가 울어도 파도는 밀려온다' -> 긍정 89.05%
'권선징악' -> 긍정 99.48%
'사랑시 고백구 행복동' -> 긍정 99.92%
'너가 좋아' -> 긍정 51.42%


## 2. 제로샷(Zero-shot-분류)
- 비지도학습
```
제로샷 분류는 기계학습 및 자연어 처리에서 개별 작업에 대한 별도의 학습(파인튜닝) 없이도 분류 작업을 수행할 수 있는 방식이다. 
분류하고자 하는 후보 레이블(candidate_labels)만 지정해 주면, 
모델이 사전에 학습한 언어 지식을 바탕으로 입력 문장이 어떤 레이블에 가장 가까운지 확률로 계산해 준다.
```

In [12]:
classifier = pipeline(
    task='zero-shot-classification',
    model='facebook/bart-large-mnli'
)

classifier(
    "I have a problem with iphone that needs to be resolved asap!!",
    candidate_labels=['phone', 'urgent', 'tablet', 'computer']
)

Device set to use cpu


{'sequence': 'I have a problem with iphone that needs to be resolved asap!!',
 'labels': ['phone', 'urgent', 'computer', 'tablet'],
 'scores': [0.6188073754310608,
  0.37655454874038696,
  0.003668452613055706,
  0.000969655578956008]}

In [13]:
classifier("This is a course about the Tranformers library.",
          candidate_labels=["education", "business", "phone"])

{'sequence': 'This is a course about the Tranformers library.',
 'labels': ['education', 'business', 'phone'],
 'scores': [0.8320797681808472, 0.09927219897508621, 0.06864806264638901]}